# Pr?ctica 1 ? Clasificaci?n de Im?genes con Redes Convolucionales
### Aprendizaje Autom?tico II - Curso 2026/2027
**Problema:** Clasificador de Residuos (7 clases: t-cardboard, t-e-waste, t-glass, t-metal, t-organic, t-plastic, t-textile).
**Objetivos:**
1. Dise?o de arquitectura convolucional en PyTorch (convoluci?n, pooling, dropout, fully connected).
2. Mecanismo de parada anticipada (Early Stopping).
3. B?squeda de hiperpar?metros manual (m?nimo 5 configuraciones).
4. T?cnicas de Data Augmentation.
5. B?squeda autom?tica de hiperpar?metros (Hyperparameter Tuning con Optuna).
6. Transferencia de conocimiento (Transfer Learning) comparando ResNet18 y MobileNetV2.


In [ ]:
import os
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 1. Carga de Datos, Robustez y Data Augmentation
Se define la clase personalizada `SafeImageFolder` para asegurar la carga ?nicamente de directorios v?lidos con im?genes. Para el conjunto de entrenamiento se aplican transformaciones de data augmentation (rotaciones, volteos, color jitter, affine) y la normalizaci?n est?ndar de ImageNet.


In [ ]:
class SafeImageFolder(ImageFolder):
    def find_classes(self, directory):
        classes, class_to_idx = super().find_classes(directory)
        valid_classes = []
        for c in classes:
            class_path = os.path.join(directory, c)
            if os.path.isdir(class_path):
                has_images = any(f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')) for f in os.listdir(class_path))
                if has_images:
                    valid_classes.append(c)
        new_class_to_idx = {cls_name: i for i, cls_name in enumerate(sorted(valid_classes))}
        return sorted(valid_classes), new_class_to_idx

def obtener_dataloaders(directorio_base='Basura_Dataset', resolucion=(224, 224), batch_size=32):
    transformaciones_train = transforms.Compose([
        transforms.Resize(resolucion),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    transformaciones_val_test = transforms.Compose([
        transforms.Resize(resolucion),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_path = os.path.join(directorio_base, 'train')
    val_path = os.path.join(directorio_base, 'val')
    test_path = os.path.join(directorio_base, 'test')

    dataset_train = SafeImageFolder(root=train_path, transform=transformaciones_train)
    dataset_val = SafeImageFolder(root=val_path, transform=transformaciones_val_test)
    dataset_test = SafeImageFolder(root=test_path, transform=transformaciones_val_test)

    dl_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
    dl_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
    dl_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)

    return dl_train, dl_val, dl_test, dataset_train.classes


## 2. Red Neuronal Convolucional Propia (CNN)
Arquitectura modular parametrizable que combina capas de convoluci?n 2D, funciones de activaci?n (ReLU o LeakyReLU), capas de reducci?n espacial MaxPool2d, regularizaci?n mediante Dropout y capas completamente conectadas (Linear).


In [ ]:
class RedConvolucionalResiduos(nn.Module):
    def __init__(self, num_clases=7, tam_entrada=96, neuronas_fc=128, tasa_dropout=0.25, usar_leaky_relu=False):
        super(RedConvolucionalResiduos, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=2)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.dropout = nn.Dropout(p=tasa_dropout)
        
        self.activacion = nn.LeakyReLU(negative_slope=0.1) if usar_leaky_relu else nn.ReLU()
        
        tam_reducido = tam_entrada // (2 ** 3)
        dimension_plana = 64 * tam_reducido * tam_reducido
        
        self.fc1 = nn.Linear(dimension_plana, neuronas_fc)
        self.fc2 = nn.Linear(neuronas_fc, num_clases)

    def forward(self, x):
        x = self.pool(self.activacion(self.conv1(x)))
        x = self.pool(self.activacion(self.conv2(x)))
        x = self.pool(self.activacion(self.conv3(x)))
        
        x = x.view(x.size(0), -1)
        x = self.activacion(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


## 3. Entrenamiento con Parada Anticipada (Early Stopping)
El algoritmo guarda en memoria una copia de los mejores pesos (`state_dict`) basada en la menor p?rdida de validaci?n (`Val Loss`). Si durante un n?mero de ?pocas consecutivas (`paciencia=5`) no hay mejora, el bucle de entrenamiento se interrumpe prematuramente para prevenir el sobreajuste.


In [ ]:
def entrenar_con_early_stopping(modelo, dl_train, dl_val, criterio, optimizador, epocas=50, paciencia=5, dispositivo=device):
    modelo = modelo.to(dispositivo)
    mejor_perdida_val = float('inf')
    mejor_epoca = 0
    mejor_estado = None
    epocas_sin_mejora = 0

    historial = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }

    tiempo_inicio = time.time()

    for epoca in range(epocas):
        modelo.train()
        perdida_train_acumulada = 0.0
        y_real_train, y_pred_train = [], []

        for imagenes, etiquetas in dl_train:
            imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)

            optimizador.zero_grad()
            salidas = modelo(imagenes)
            perdida = criterio(salidas, etiquetas)
            perdida.backward()
            optimizador.step()

            perdida_train_acumulada += perdida.item() * imagenes.size(0)
            y_pred_train.extend(salidas.argmax(dim=1).cpu().numpy())
            y_real_train.extend(etiquetas.cpu().numpy())

        train_loss = perdida_train_acumulada / len(dl_train.dataset)
        train_acc = accuracy_score(y_real_train, y_pred_train)

        modelo.eval()
        perdida_val_acumulada = 0.0
        y_real_val, y_pred_val = [], []

        with torch.no_grad():
            for imagenes, etiquetas in dl_val:
                imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)
                salidas = modelo(imagenes)
                perdida = criterio(salidas, etiquetas)

                perdida_val_acumulada += perdida.item() * imagenes.size(0)
                y_pred_val.extend(salidas.argmax(dim=1).cpu().numpy())
                y_real_val.extend(etiquetas.cpu().numpy())

        val_loss = perdida_val_acumulada / len(dl_val.dataset)
        val_acc = accuracy_score(y_real_val, y_pred_val)

        historial['train_loss'].append(train_loss)
        historial['val_loss'].append(val_loss)
        historial['train_acc'].append(train_acc)
        historial['val_acc'].append(val_acc)

        print(f"Epoca [{epoca+1:2d}/{epocas}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_loss < mejor_perdida_val:
            mejor_perdida_val = val_loss
            mejor_epoca = epoca + 1
            mejor_estado = copy.deepcopy(modelo.state_dict())
            epocas_sin_mejora = 0
            print("   -> Mejora en validacion: checkpoint guardado")
        else:
            epocas_sin_mejora += 1
            print(f"   -> Sin mejora ({epocas_sin_mejora}/{paciencia})")
            if epocas_sin_mejora >= paciencia:
                print(f"\nParada temprana activada en epoca {epoca+1}. Punto optimo recuperado de la epoca {mejor_epoca} (Val Loss: {mejor_perdida_val:.4f})")
                break

    tiempo_total = time.time() - tiempo_inicio
    print(f"Tiempo total de entrenamiento: {tiempo_total:.2f} segundos")
    
    if mejor_estado is not None:
        modelo.load_state_dict(mejor_estado)
        
    return modelo, historial, mejor_epoca


## 4. Funciones de Evaluaci?n y Visualizaci?n
Se implementa la generaci?n obligatoria de gr?ficas de p?rdida y precisi?n lado a lado, y la matriz de confusi?n mediante la librer?a Seaborn.


In [ ]:
def graficar_metricas(historial, mejor_epoca, titulo='Rendimiento del Modelo'):
    epocas = range(1, len(historial['train_loss']) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(epocas, historial['train_loss'], label='Train Loss', color='#1f77b4', lw=2)
    ax1.plot(epocas, historial['val_loss'], label='Val Loss', color='#ff7f0e', lw=2)
    ax1.axvline(mejor_epoca, color='red', linestyle='--', label=f'Mejor Epoca ({mejor_epoca})')
    ax1.set_title(f'{titulo} - Perdida', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epocas')
    ax1.set_ylabel('CrossEntropy Loss')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epocas, historial['train_acc'], label='Train Acc', color='#2ca02c', lw=2)
    ax2.plot(epocas, historial['val_acc'], label='Val Acc', color='#d62728', lw=2)
    ax2.axvline(mejor_epoca, color='red', linestyle='--', label=f'Mejor Epoca ({mejor_epoca})')
    ax2.set_title(f'{titulo} - Precision (Accuracy)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epocas')
    ax2.set_ylabel('Accuracy')
    ax2.legend(loc='lower right')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def evaluar_en_test(modelo, dl_test, clases, dispositivo=device):
    modelo = modelo.to(dispositivo)
    modelo.eval()
    y_real, y_pred = [], []
    
    with torch.no_grad():
        for imagenes, etiquetas in dl_test:
            imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)
            salidas = modelo(imagenes)
            y_pred.extend(salidas.argmax(dim=1).cpu().numpy())
            y_real.extend(etiquetas.cpu().numpy())
            
    acc = accuracy_score(y_real, y_pred)
    print(f"Accuracy en Test: {acc*100:.2f}%\n")
    print(classification_report(y_real, y_pred, target_names=clases, digits=4))
    
    cm = confusion_matrix(y_real, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=clases, yticklabels=clases)
    plt.title('Matriz de Confusion en Test', fontsize=12, fontweight='bold')
    plt.xlabel('Prediccion')
    plt.ylabel('Etiqueta Real')
    plt.tight_layout()
    plt.show()
    return acc, cm


## 5. B?squeda Manual de Hiperpar?metros
Se exploran de forma manual y secuencial 5 configuraciones distintas en la red convolucional, variando la resoluci?n de entrada, la dimensi?n de las capas densas, la regularizaci?n por dropout y las funciones de activaci?n.


In [ ]:
dl_train_96, dl_val_96, dl_test_96, clases = obtener_dataloaders(resolucion=(96, 96), batch_size=32)

modelo_manual_1 = RedConvolucionalResiduos(num_clases=7, tam_entrada=96, neuronas_fc=128, tasa_dropout=0.25, usar_leaky_relu=False)
criterio = nn.CrossEntropyLoss()
optimizador_1 = optim.Adam(modelo_manual_1.parameters(), lr=0.001)

modelo_manual_1, hist_1, epoca_opt_1 = entrenar_con_early_stopping(
    modelo_manual_1, dl_train_96, dl_val_96, criterio, optimizador_1, epocas=30, paciencia=5
)
graficar_metricas(hist_1, epoca_opt_1, titulo='Prueba 1: CNN Base (96x96, FC 128)')
acc_test_1, _ = evaluar_en_test(modelo_manual_1, dl_test_96, clases)


## 6. Elemento Opcional: B?squeda Autom?tica de Hiperpar?metros (Tuning con Optuna)
Se utiliza optimizaci?n bayesiana con el algoritmo Tree-structured Parzen Estimator (TPE) mediante Optuna para encontrar la combinaci?n ?ptima de hiperpar?metros (learning rate, tasa de dropout, n?mero de neuronas densas y funci?n de activaci?n).


In [ ]:
try:
    import optuna
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo_optuna(trial):
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    dropout = trial.suggest_float('dropout', 0.15, 0.50, step=0.05)
    neuronas_fc = trial.suggest_categorical('neuronas_fc', [64, 128, 256])
    usar_leaky = trial.suggest_categorical('usar_leaky', [True, False])
    
    modelo_trial = RedConvolucionalResiduos(
        num_clases=7, tam_entrada=96, neuronas_fc=neuronas_fc, tasa_dropout=dropout, usar_leaky_relu=usar_leaky
    ).to(device)
    
    criterio_trial = nn.CrossEntropyLoss()
    optimizador_trial = optim.Adam(modelo_trial.parameters(), lr=lr)
    
    mejor_acc = 0.0
    for epoca in range(12):
        modelo_trial.train()
        for imagenes, etiquetas in dl_train_96:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            optimizador_trial.zero_grad()
            salidas = modelo_trial(imagenes)
            perdida = criterio_trial(salidas, etiquetas)
            perdida.backward()
            optimizador_trial.step()
            
        modelo_trial.eval()
        y_real, y_pred = [], []
        with torch.no_grad():
            for imagenes, etiquetas in dl_val_96:
                imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
                salidas = modelo_trial(imagenes)
                y_pred.extend(salidas.argmax(dim=1).cpu().numpy())
                y_real.extend(etiquetas.cpu().numpy())
        acc_val = accuracy_score(y_real, y_pred)
        if acc_val > mejor_acc:
            mejor_acc = acc_val
            
    return mejor_acc

estudio = optuna.create_study(direction='maximize')
estudio.optimize(objetivo_optuna, n_trials=10, timeout=600)

print(f"Mejor Accuracy en Validacion alcanzado por Optuna: {estudio.best_value*100:.2f}%")
print("Mejores Hiperparametros encontrados:")
for clave, valor in estudio.best_params.items():
    print(f"  {clave}: {valor}")


## 7. Elemento Opcional: Transferencia de Conocimiento (Transfer Learning)
Se eval?an dos arquitecturas pre-entrenadas del estado del arte:
1. **ResNet-18:** Red residual profunda con conexiones de salto para capturar caracter?sticas complejas.
2. **MobileNetV2:** Red optimizada con convoluciones invertidas separables en profundidad.


In [ ]:
dl_train_224, dl_val_224, dl_test_224, clases = obtener_dataloaders(resolucion=(224, 224), batch_size=32)

modelo_resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_caracteristicas = modelo_resnet18.fc.in_features
modelo_resnet18.fc = nn.Linear(num_caracteristicas, len(clases))

criterio = nn.CrossEntropyLoss()
optimizador_resnet = optim.Adam(modelo_resnet18.parameters(), lr=0.0005)

modelo_resnet18, hist_resnet, epoca_resnet = entrenar_con_early_stopping(
    modelo_resnet18, dl_train_224, dl_val_224, criterio, optimizador_resnet, epocas=35, paciencia=5
)
graficar_metricas(hist_resnet, epoca_resnet, titulo='Transfer Learning: ResNet18')
acc_test_resnet, _ = evaluar_en_test(modelo_resnet18, dl_test_224, clases)


In [ ]:
modelo_mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
num_caracteristicas_mb = modelo_mobilenet.classifier[1].in_features
modelo_mobilenet.classifier[1] = nn.Linear(num_caracteristicas_mb, len(clases))

optimizador_mb = optim.Adam(modelo_mobilenet.parameters(), lr=0.0005)

modelo_mobilenet, hist_mb, epoca_mb = entrenar_con_early_stopping(
    modelo_mobilenet, dl_train_224, dl_val_224, criterio, optimizador_mb, epocas=35, paciencia=5
)
graficar_metricas(hist_mb, epoca_mb, titulo='Transfer Learning: MobileNetV2')
acc_test_mb, _ = evaluar_en_test(modelo_mobilenet, dl_test_224, clases)
